<a href="https://colab.research.google.com/github/prasanna-venkatesh-m/sample-transformer-architecture/blob/main/TransformerArchitecture.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import tensorflow as tf
import numpy as np
import math

In [ ]:
texts = [
    "cat sat on mat",
    "dog sat on rug",
    "cat chased mouse",
    "dog chased cat",
    "cat sat on rug",
    "dog chased mouse",
    "cat chased cat",
    "dog sat on mat",
    "mouse ran from cat",
    "cat ran after mouse",
    "dog ran after cat",
    "mouse sat on mat",
    "cat sat on cat",
    "dog chased dog"
]

In [ ]:
MAX_TOKENS = 1000
SEQ_LEN = 6

vectorizer = tf.keras.layers.TextVectorization(
    max_tokens = MAX_TOKENS,
    output_sequence_length = SEQ_LEN
)

vectorizer.adapt(texts)

In [ ]:
class PositionalEmbedding(tf.keras.layers.Layer):
  def __init__(self, d_model, max_len=100):
    super().__init__()

    pe = np.zeros((max_len, d_model))
    position = np.arange(max_len)[:, np.newaxis]
    div_term = np.exp(
        np.arange(0, d_model, 2) * (-math.log(10000.0) / d_model)
    )

    pe[:, 0::2] = np.sin(position * div_term)
    pe[:, 1::2] = np.cos(position * div_term)

    self.pe = tf.constant(pe, dtype = np.float32)

  def call(self, x):
    seq_len = tf.shape(x)[1]
    return x + self.pe[:seq_len]

In [ ]:
class SingleHeadAttention(tf.keras.layers.Layer):
  def __init__(self, d_model):
    super().__init__()
    self.Wq = tf.keras.layers.Dense(d_model)
    self.Wk = tf.keras.layers.Dense(d_model)
    self.Wv = tf.keras.layers.Dense(d_model)

  def call(self, x):
    Q = self.Wq(x)
    K = self.Wk(x)
    V = self.Wv(x)

    scores = tf.matmul(Q,K, transpose_b=True)
    scores /= tf.math.sqrt(tf.cast(tf.shape(K)[-1], tf.float32))

    weights = tf.nn.softmax(scores, axis=-1)
    return tf.matmul(weights, V)

In [ ]:
class FeedForward(tf.keras.layers.Layer):
  def __init__(self, d_model, hidden_dim):
    super().__init__()
    self.fc1 = tf.keras.layers.Dense(hidden_dim, activation = "relu")
    self.fc2 = tf.keras.layers.Dense(d_model)

  def call(self, x):
    return self.fc2(self.fc1(x))

In [ ]:
class TransformerBlock(tf.keras.layers.Layer):
  def __init__(self, d_model, hidden_dim):
    super().__init__()
    self.attention = SingleHeadAttention(d_model)
    self.norm1 = tf.keras.layers.LayerNormalization()
    self.ffn = FeedForward(d_model, hidden_dim)
    self.norm2 = tf.keras.layers.LayerNormalization()

  def call(self, x):
    attn_out = self.attention(x)
    x = self.norm1(x + attn_out)

    ffn_out = self.ffn(x)
    x = self.norm2(x + ffn_out)

    return x

In [ ]:
class MiniTransformer(tf.keras.Model):
    def __init__(self, vocab_size, d_model, hidden_dim):
        super().__init__()

        self.embedding = tf.keras.layers.Embedding(
            input_dim=vocab_size,
            output_dim=d_model
        )

        self.positional_encoding = PositionalEmbedding(d_model)
        self.transformer = TransformerBlock(d_model, hidden_dim)
        self.output_layer = tf.keras.layers.Dense(vocab_size)

    def call(self, x):
        x = self.embedding(x)
        x = self.positional_encoding(x)
        x = self.transformer(x)
        return self.output_layer(x)

In [ ]:
VOCAB_SIZE = vectorizer.vocabulary_size()
D_MODEL = 32
HIDDEN_DIM = 64

model = MiniTransformer(VOCAB_SIZE, D_MODEL, HIDDEN_DIM)

In [ ]:
tokens = vectorizer(texts)
inputs = tokens[:, :-1]   # all except last token
targets = tokens[:, 1:]   # all except first token
logits = model(tokens)

print("Tokens shape:", tokens.shape)
print("Logits shape:", logits.shape)


Tokens shape: (14, 6)
Logits shape: (14, 6, 13)


In [ ]:
targets = tokens  # language modeling (predict next token)

loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
optimizer = tf.keras.optimizers.Adam()

with tf.GradientTape() as tape:
    logits = model(tokens)
    loss = loss_fn(targets, logits)

grads = tape.gradient(loss, model.trainable_variables)
optimizer.apply_gradients(zip(grads, model.trainable_variables))

print("Loss:", loss.numpy())

Loss: 2.6239898


In [ ]:
# Create inputs and targets
tokens = vectorizer(texts)
inputs = tokens[:, :-1]   # all tokens except last
targets = tokens[:, 1:]   # all tokens except first

# Convert to tf.data.Dataset for batching and shuffling
BATCH_SIZE = 4
dataset = tf.data.Dataset.from_tensor_slices((inputs, targets))
dataset = dataset.shuffle(buffer_size=len(inputs)).batch(BATCH_SIZE).repeat()

# Training parameters
EPOCHS = 200
STEPS_PER_EPOCH = len(inputs) // BATCH_SIZE

# Optimizer and loss
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
optimizer = tf.keras.optimizers.Adam()

# Training loop
for epoch in range(EPOCHS):
    epoch_loss = 0.0
    for step, (x_batch, y_batch) in enumerate(dataset.take(STEPS_PER_EPOCH)):
        with tf.GradientTape() as tape:
            logits = model(x_batch)               # shape: (batch, seq_len, vocab_size)
            loss = loss_fn(y_batch, logits)      # shape matches targets

        grads = tape.gradient(loss, model.trainable_variables)
        optimizer.apply_gradients(zip(grads, model.trainable_variables))
        epoch_loss += loss.numpy()

    if epoch % 20 == 0:
        print(f"Epoch {epoch}, Loss: {epoch_loss / STEPS_PER_EPOCH:.4f}")


Epoch 0, Loss: 2.3370
Epoch 20, Loss: 0.9411
Epoch 40, Loss: 0.2787
Epoch 60, Loss: 0.1761
Epoch 80, Loss: 0.0633
Epoch 100, Loss: 0.0222
Epoch 120, Loss: 0.0117
Epoch 140, Loss: 0.0079
Epoch 160, Loss: 0.0058
Epoch 180, Loss: 0.0046


In [ ]:
vocab = vectorizer.get_vocabulary()
prompt = "cat sat"
prompt_tokens = vectorizer([prompt])[:, :-1]  # shape: (1, seq_len-1)

logits = model(prompt_tokens)
pred_token = tf.argmax(logits[:, -1, :], axis=-1).numpy()[0]

print("Prompt:", prompt)
print("Predicted next word:", vocab[pred_token])

Prompt: cat sat
Predicted next word: 


In [ ]:
prompt = "dog"
prompt_tokens = vectorizer([prompt])[:, :-1]
logits = model(prompt_tokens)
pred_token = tf.argmax(logits[:, -1, :], axis=-1).numpy()[0]
vocab = vectorizer.get_vocabulary()

print("Prompt:", prompt)
print("Predicted next word:", vocab[pred_token])


Prompt: dog
Predicted next word: 


In [ ]:
BATCH_SIZE = 4
dataset = tf.data.Dataset.from_tensor_slices((inputs, targets))
dataset = dataset.shuffle(len(inputs)).repeat(500).batch(BATCH_SIZE)

optimizer = tf.keras.optimizers.Adam()
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)

for step, (x_batch, y_batch) in enumerate(dataset):
    with tf.GradientTape() as tape:
        logits = model(x_batch)
        loss = loss_fn(y_batch, logits)

    grads = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(grads, model.trainable_variables))

    if step % 50 == 0:
        print(f"Step {step}, Loss: {loss.numpy():.4f}")

    if step > 2000:  # stop after 2000 steps
        break

Step 0, Loss: 0.0041
Step 50, Loss: 0.0007
Step 100, Loss: 0.0004
Step 150, Loss: 0.0002
Step 200, Loss: 0.0001
Step 250, Loss: 0.0001
Step 300, Loss: 0.0001
Step 350, Loss: 0.0001
Step 400, Loss: 0.0001
Step 450, Loss: 0.0001
Step 500, Loss: 0.0000
Step 550, Loss: 0.0000
Step 600, Loss: 0.0000
Step 650, Loss: 0.0000
Step 700, Loss: 0.0000
Step 750, Loss: 0.0000
Step 800, Loss: 0.0000
Step 850, Loss: 0.0000
Step 900, Loss: 0.0000
Step 950, Loss: 0.0000
Step 1000, Loss: 0.0000
Step 1050, Loss: 0.0000
Step 1100, Loss: 0.0000
Step 1150, Loss: 0.0000
Step 1200, Loss: 0.0000
Step 1250, Loss: 0.0000
Step 1300, Loss: 0.0000
Step 1350, Loss: 0.0000
Step 1400, Loss: 0.0000
Step 1450, Loss: 0.0000
Step 1500, Loss: 0.0000
Step 1550, Loss: 0.0000
Step 1600, Loss: 0.0000
Step 1650, Loss: 0.0000
Step 1700, Loss: 0.0000


In [ ]:
prompt = "cat sat"
prompt_tokens = vectorizer([prompt])[:, :-1]
logits = model(prompt_tokens)
pred_token = tf.argmax(logits[:, -1, :], axis=-1).numpy()[0]
vocab = vectorizer.get_vocabulary()

print("Prompt:", prompt)
print("Predicted next word:", vocab[pred_token])

Prompt: cat sat
Predicted next word: 


AI CODE

In [ ]:
import tensorflow as tf
import numpy as np
import math

# --- Tiny dataset ---
texts = [
    "cat sat on mat",
    "dog sat on rug",
    "cat chased mouse",
    "dog chased cat",
    "cat sat on rug",
    "dog chased mouse",
    "cat chased cat",
    "dog sat on mat",
    "mouse ran from cat",
    "cat ran after mouse",
    "dog ran after cat",
    "mouse sat on mat",
    "cat sat on cat",
    "dog chased dog"
]

MAX_TOKENS = 1000
SEQ_LEN = 6

# --- Vectorizer ---
vectorizer = tf.keras.layers.TextVectorization(
    max_tokens = MAX_TOKENS,
    output_sequence_length = SEQ_LEN
)
vectorizer.adapt(texts)

# --- Positional Embedding ---
class PositionalEmbedding(tf.keras.layers.Layer):
    def __init__(self, d_model, max_len=100):
        super().__init__()
        pe = np.zeros((max_len, d_model))
        position = np.arange(max_len)[:, np.newaxis]
        div_term = np.exp(np.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = np.sin(position * div_term)
        pe[:, 1::2] = np.cos(position * div_term)
        self.pe = tf.constant(pe, dtype=tf.float32)
    def call(self, x):
        seq_len = tf.shape(x)[1]
        return x + self.pe[:seq_len]

# --- Single-head attention ---
class SingleHeadAttention(tf.keras.layers.Layer):
    def __init__(self, d_model):
        super().__init__()
        self.Wq = tf.keras.layers.Dense(d_model)
        self.Wk = tf.keras.layers.Dense(d_model)
        self.Wv = tf.keras.layers.Dense(d_model)
    def call(self, x):
        Q = self.Wq(x)
        K = self.Wk(x)
        V = self.Wv(x)
        scores = tf.matmul(Q, K, transpose_b=True)
        scores /= tf.math.sqrt(tf.cast(tf.shape(K)[-1], tf.float32))
        weights = tf.nn.softmax(scores, axis=-1)
        return tf.matmul(weights, V)

# --- Feed-forward ---
class FeedForward(tf.keras.layers.Layer):
    def __init__(self, d_model, hidden_dim):
        super().__init__()
        self.fc1 = tf.keras.layers.Dense(hidden_dim, activation='relu')
        self.fc2 = tf.keras.layers.Dense(d_model)
    def call(self, x):
        return self.fc2(self.fc1(x))

# --- Transformer block ---
class TransformerBlock(tf.keras.layers.Layer):
    def __init__(self, d_model, hidden_dim):
        super().__init__()
        self.attention = SingleHeadAttention(d_model)
        self.norm1 = tf.keras.layers.LayerNormalization()
        self.ffn = FeedForward(d_model, hidden_dim)
        self.norm2 = tf.keras.layers.LayerNormalization()
    def call(self, x):
        attn_out = self.attention(x)
        x = self.norm1(x + attn_out)
        ffn_out = self.ffn(x)
        x = self.norm2(x + ffn_out)
        return x

# --- Mini Transformer ---
class MiniTransformer(tf.keras.Model):
    def __init__(self, vocab_size, d_model, hidden_dim):
        super().__init__()
        self.embedding = tf.keras.layers.Embedding(input_dim=vocab_size, output_dim=d_model)
        self.positional_encoding = PositionalEmbedding(d_model)
        self.transformer = TransformerBlock(d_model, hidden_dim)
        self.output_layer = tf.keras.layers.Dense(vocab_size)
    def call(self, x):
        x = self.embedding(x)
        x = self.positional_encoding(x)
        x = self.transformer(x)
        return self.output_layer(x)

# --- Model ---
VOCAB_SIZE = vectorizer.vocabulary_size()
D_MODEL = 16    # smaller for tiny dataset
HIDDEN_DIM = 32
model = MiniTransformer(VOCAB_SIZE, D_MODEL, HIDDEN_DIM)

# --- Prepare inputs/targets ---
tokens = vectorizer(texts)
inputs = tokens[:, :-1]   # all except last
targets = tokens[:, 1:]   # all except first

# --- Dataset with repetition to memorize ---
BATCH_SIZE = 4
dataset = tf.data.Dataset.from_tensor_slices((inputs, targets))
dataset = dataset.shuffle(len(inputs)).repeat(1000).batch(BATCH_SIZE)

# --- Optimizer & loss ---
optimizer = tf.keras.optimizers.Adam(0.01)
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)

# --- Training loop ---
for step, (x_batch, y_batch) in enumerate(dataset):
    with tf.GradientTape() as tape:
        logits = model(x_batch)
        loss = loss_fn(y_batch, logits)
    grads = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(grads, model.trainable_variables))

    if step % 50 == 0:
        print(f"Step {step}, Loss: {loss.numpy():.4f}")

    if step >= 2000:  # stop after 2000 steps
        break

# --- Prediction ---
prompt = "cat sat"
prompt_tokens = vectorizer([prompt])[:, :-1]  # remove last token
logits = model(prompt_tokens)
pred_token = tf.argmax(logits[:, -1, :], axis=-1).numpy()[0]
vocab = vectorizer.get_vocabulary()

print("\nPrompt:", prompt)
print("Predicted next word:", vocab[pred_token])


Step 0, Loss: 2.7752
Step 50, Loss: 0.3145
Step 100, Loss: 0.0737
Step 150, Loss: 0.0062
Step 200, Loss: 0.0044
Step 250, Loss: 0.0020
Step 300, Loss: 0.0017
Step 350, Loss: 0.0008
Step 400, Loss: 0.0006
Step 450, Loss: 0.0007
Step 500, Loss: 0.0007
Step 550, Loss: 0.0004
Step 600, Loss: 0.0003
Step 650, Loss: 0.0004
Step 700, Loss: 0.0004
Step 750, Loss: 0.0003
Step 800, Loss: 0.0002
Step 850, Loss: 0.0002
Step 900, Loss: 0.0002
Step 950, Loss: 0.0002
Step 1000, Loss: 0.0001
Step 1050, Loss: 0.0001
Step 1100, Loss: 0.0001
Step 1150, Loss: 0.0002
Step 1200, Loss: 0.0001
Step 1250, Loss: 0.0001
Step 1300, Loss: 0.0001
Step 1350, Loss: 0.0001
Step 1400, Loss: 0.0001
Step 1450, Loss: 0.0001
Step 1500, Loss: 0.0001
Step 1550, Loss: 0.0001
Step 1600, Loss: 0.0001
Step 1650, Loss: 0.0001
Step 1700, Loss: 0.0001
Step 1750, Loss: 0.0001
Step 1800, Loss: 0.0000
Step 1850, Loss: 0.0001
Step 1900, Loss: 0.0000
Step 1950, Loss: 0.0001
Step 2000, Loss: 0.0000

Prompt: cat sat
Predicted next word: 


In [ ]:
import tensorflow as tf

# Tiny dataset
texts = [
    "cat sat on mat",
    "cat sat on rug",
    "cat sat on cat",
    "dog sat on rug",
    "dog sat on mat",
    "dog chased cat",
    "cat chased mouse",
    "cat chased cat",
    "mouse ran from cat",
    "cat ran after mouse",
] * 50

# Vectorize text
vectorizer = tf.keras.layers.TextVectorization(output_sequence_length=5)
vectorizer.adapt(texts)
tokens = vectorizer(texts)
inputs = tokens[:, :-1]  # all except last token
targets = tokens[:, 1:]  # all except first token

# Repeat dataset to give more training signal
inputs = tf.tile(inputs, [20, 1])
targets = tf.tile(targets, [20, 1])

# Build model
model = tf.keras.Sequential([
    tf.keras.layers.Embedding(input_dim=vectorizer.vocabulary_size(), output_dim=32),
    tf.keras.layers.LSTM(64, return_sequences=True),
    tf.keras.layers.Dense(vectorizer.vocabulary_size())
])

model.compile(optimizer='adam', loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True))

# Train
model.fit(inputs, targets, epochs=300, verbose=0)

# Prediction
prompt = "cat sat"
prompt_tokens = vectorizer([prompt])[:, :-1]
logits = model(prompt_tokens)
pred_token = tf.argmax(logits[:, -1, :], axis=-1).numpy()[0]
vocab = vectorizer.get_vocabulary()
print("Prompt:", prompt)
print("Predicted next word:", vocab[pred_token])


Prompt: cat sat
Predicted next word: 
